# 3.3b GenAIOps: prompts, agent versions, tests and RAG refresh

**Story:** The SafeDrive assistant works. Now it must be **operated**: prompts change, the agent gets
new instructions, policy documents are updated every week and a fine-tuned model may replace the base
model. Nothing may reach customers without a version, a test and a way back.

**Task 3.3** Implement automated orchestration and CI/CD pipelines for MLOps and AI workloads.

| Skill | What | Status |
|---|---|---|
| 3.3.7 | Manage prompts (Bedrock Prompt Management) | ✅ |
| 3.3.9 | AI model testing frameworks, including prompt testing | ✅ |
| 3.3.8 | Automated agent deployment pipelines and agent version management | ✅ |
| 3.3.10 | FM deployment automation with fine-tuned model versioning | 🔒 |
| 3.3.11 | Pipeline orchestration for RAG updates and knowledge base refresh | ✅ |

## Key points for the exam

- Treat **prompts, agents, knowledge bases and models as versioned artifacts**, like code.
- **Prompt Management**: a prompt has a working **DRAFT** and immutable numbered **versions**.
  **Variants** are alternative wordings you compare before saving. Apps call a **version ARN**, never the draft.
- **Bedrock Agents**: DRAFT → `prepare_agent` → **version** (immutable) → **alias** (pointer).
  Promote and roll back by **repointing the alias**. Client code never changes.
- **AgentCore Runtime**: every update creates a new runtime **version**. An **endpoint** points to a
  version (`DEFAULT` follows the latest).
- **Prompt regression tests** run in CI (CodeBuild) on every change to a prompt, model or KB: golden
  questions, format checks, refusal checks, latency and token budgets, LLM-as-a-judge.
- LLM output is not deterministic: test with `temperature=0`, assert on facts and thresholds, not on
  exact strings.
- **RAG refresh**: S3 event → EventBridge → Lambda → `StartIngestionJob`, or a schedule. Evaluate
  before you promote.
- Keep the **model ID in configuration** (SSM Parameter Store, AppConfig), so promoting a fine-tuned
  model is a config change with an easy rollback.

| Artifact | Version mechanism | Promote / roll back |
|---|---|---|
| Prompt | `create_prompt_version` | app config points to the version ARN |
| Bedrock agent | version created with an alias | `update_agent_alias` routing |
| AgentCore runtime | version per `update_agent_runtime` | `update_agent_runtime_endpoint` |
| Fine-tuned model | custom model name + tags | SSM parameter with the deployment ARN |
| Knowledge base | documents in S3 (S3 Versioning) + ingestion jobs | re-sync, or switch to a second KB |

## Setup

In [ ]:
import sys; sys.path.append("..")
import json, os, time, uuid
import pandas as pd
from common import utils as U, catchup as C
os.makedirs("scripts", exist_ok=True)
print(U.REGION, U.bucket())

Catch-up: raw data, the Bedrock agent (3.1b) and the Knowledge Base (3.2b).

In [ ]:
C.ensure_raw_data()
agent_id, agent_alias_id = C.ensure_agent()
kb_id = C.ensure_kb()
ds_id = U.get_state("kb_data_source_id")
print("agent:", agent_id, agent_alias_id, "| kb:", kb_id, ds_id)

In [ ]:
agent = U.client("bedrock-agent")
rt = U.client("bedrock-agent-runtime")
brt = U.client("bedrock-runtime")
bedrock = U.client("bedrock")
lam = U.client("lambda")
ROLE = U.role_arn()
qa = U.read_jsonl(f"{C.RAW}/qa_eval.jsonl")
print(len(qa), "golden questions")

## 3.3.7 Manage prompts with Bedrock Prompt Management

- A managed prompt stores the **template**, its **variables** (`{{name}}`), the **model** and the
  **inference settings** together.
- **Variants** are alternative wordings (or models) of the same prompt. The console compares up to
  three side by side and saves the winner. Through the API a prompt stores **one variant** per
  version, so you compare candidates in code and publish the winner as a new version.
- `create_prompt_version` takes a snapshot of the draft. Versions cannot be edited.
- The Converse API accepts a **prompt ARN as `modelId`** and fills the variables from `promptVariables`.

Two candidate variants for the same job. Both must return JSON, which makes automated tests easy.

In [ ]:
RULES = ('Answer the question using only the policy text. Reply with JSON only: '
         '{"answer": "<text>", "grounded": true or false}. Set grounded to false and say you can only '
         'help with SafeDrive policy questions if the policy text does not contain the answer.')
CONCISE = RULES + "\nAnswer in one short sentence.\n\nPolicy text:\n{{context}}\n\nQuestion: {{question}}"
FRIENDLY = RULES + ("\nUse a warm, friendly tone and at most two sentences."
                    "\n\nPolicy text:\n{{context}}\n\nQuestion: {{question}}")

def variant(name, text, temperature):
    return {"name": name, "templateType": "TEXT", "modelId": U.TEXT_MODEL,
            "inferenceConfiguration": {"text": {"temperature": temperature, "maxTokens": 300}},
            "templateConfiguration": {"text": {
                "text": text, "inputVariables": [{"name": "context"}, {"name": "question"}]}}}

print(CONCISE)

Create the prompt with the `concise` variant (reuse it if it exists). `defaultVariant` names the
variant that is used when the prompt is invoked.

In [ ]:
PROMPT_NAME = "safedrive-policy-answer"
found = [p for p in agent.list_prompts(maxResults=100)["promptSummaries"] if p["name"] == PROMPT_NAME]
if found:
    prompt_id = found[0]["id"]
else:
    prompt_id = agent.create_prompt(
        name=PROMPT_NAME, description="Answer a policy question from retrieved context",
        defaultVariant="concise", variants=[variant("concise", CONCISE, 0.0)],
        tags={t["Key"]: t["Value"] for t in U.TAGS})["id"]
U.save_state(prompt_id=prompt_id)
print("prompt id:", prompt_id)

Read the draft back. Check the variant, its variables and the model stored with it.

In [ ]:
draft = agent.get_prompt(promptIdentifier=prompt_id)
print("version:", draft["version"], "| default variant:", draft["defaultVariant"])
for v in draft["variants"]:
    names = [i["name"] for i in v["templateConfiguration"]["text"]["inputVariables"]]
    print(" -", v["name"], v["modelId"], v["inferenceConfiguration"]["text"], names)

Publish **version 1**: an immutable snapshot of the draft.

In [ ]:
v1 = agent.create_prompt_version(promptIdentifier=prompt_id, description="first release")
prompt_arn_v1 = v1["arn"]
print(v1["version"], prompt_arn_v1)

Invoke the version through **Converse**: the prompt ARN goes in `modelId`, the values in
`promptVariables`. Model and inference settings come from the prompt, so you do not pass them.

In [ ]:
item = qa[0]
context = U.read_bytes(f"{C.RAW}/policy_docs/{item['source']}").decode()
resp = brt.converse(modelId=prompt_arn_v1,
                    promptVariables={"context": {"text": context}, "question": {"text": item["question"]}})
print(resp["output"]["message"]["content"][0]["text"])
print(resp["usage"], resp["metrics"])

Compare the two **variants** on the same question before changing anything. We render each template
ourselves, which is what the console's "Compare variants" page does.

In [ ]:
def run_variant(v, context, question):
    text = v["templateConfiguration"]["text"]["text"].replace("{{context}}", context).replace("{{question}}", question)
    cfg = v["inferenceConfiguration"]["text"]
    r = brt.converse(modelId=v["modelId"], messages=[{"role": "user", "content": [{"text": text}]}],
                     inferenceConfig={"temperature": cfg["temperature"], "maxTokens": cfg["maxTokens"]})
    return {"variant": v["name"], "output_tokens": r["usage"]["outputTokens"],
            "latency_ms": r["metrics"]["latencyMs"], "answer": r["output"]["message"]["content"][0]["text"]}

pd.set_option("display.max_colwidth", 200)
candidates = [variant("concise", CONCISE, 0.0), variant("friendly", FRIENDLY, 0.3)]
pd.DataFrame([run_variant(v, context, item["question"]) for v in candidates])

Say the friendly variant wins. Save it as the draft and publish **version 2**. `update_prompt`
replaces the whole draft.

In [ ]:
agent.update_prompt(promptIdentifier=prompt_id, name=PROMPT_NAME, defaultVariant="friendly",
                    description="Answer a policy question from retrieved context",
                    variants=[variant("friendly", FRIENDLY, 0.3)])
v2 = agent.create_prompt_version(promptIdentifier=prompt_id, description="friendly variant")
prompt_arn_v2 = v2["arn"]
print(v2["version"], prompt_arn_v2)

List all versions of the prompt. Rollback = point the application back to the older version ARN.

In [ ]:
versions = agent.list_prompts(promptIdentifier=prompt_id)["promptSummaries"]
pd.DataFrame(versions)[["name", "version", "updatedAt"]]

## 3.3.9 Prompt regression testing

- A prompt or model change can silently break answers. A **regression suite** catches that before release.
- Typical assertions:

| Check | Example here |
|---|---|
| Contains the expected fact | every number in the reference answer appears in the answer |
| Output format | reply parses as JSON with the keys `answer` and `grounded` |
| Refusal / safety | off-topic question gives `grounded: false` |
| Budget | latency and output tokens under a limit |
| Quality score | an LLM judge rates correctness, the mean must be ≥ 4 of 5 |

- Managed alternatives: **Bedrock model evaluation jobs** (automatic, LLM-as-a-judge, human) and
  RAG evaluation jobs (3.2b). The same tests should run again when the **model** changes.

The test suite as a normal **pytest** file. It reads the prompt ARN from the environment variable
`PROMPT_ARN`, so the same file tests any prompt version in the notebook and in CodeBuild.

In [ ]:
%%writefile scripts/test_prompt_regression.py
"""Prompt regression tests for the SafeDrive policy-answer prompt (run with pytest)."""
import functools
import json
import os
import re
import sys

import pytest

sys.path.append(os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", ".."))
from common import catchup as C, utils as U  # noqa: E402

brt = U.client("bedrock-runtime")
PROMPT_ARN = os.environ["PROMPT_ARN"]
GOLDEN = U.read_jsonl(f"{C.RAW}/qa_eval.jsonl")[:8]
MAX_LATENCY_MS, MAX_OUTPUT_TOKENS, MIN_JUDGE_SCORE = 8000, 200, 4.0


@functools.lru_cache(maxsize=None)
def ask(question, source):
    """Call the managed prompt once per question; return (parsed JSON or None, raw text, response)."""
    context = U.read_bytes(f"{C.RAW}/policy_docs/{source}").decode()
    resp = brt.converse(modelId=PROMPT_ARN,
                        promptVariables={"context": {"text": context}, "question": {"text": question}})
    text = resp["output"]["message"]["content"][0]["text"]
    try:
        parsed = json.loads(text[text.index("{"): text.rindex("}") + 1])
    except ValueError:
        parsed = None
    return parsed, text, resp

Two helpers, appended to the same file (`-a`): extract the numbers of a text, and an LLM judge.

In [ ]:
%%writefile -a scripts/test_prompt_regression.py


def numbers(text):
    return {float(n) for n in re.findall(r"\d+(?:\.\d+)?", text.replace(",", ""))}


def judge(question, reference, answer):
    """LLM-as-a-judge: 1 (wrong) to 5 (fully correct) against the reference answer."""
    prompt = (f"Question: {question}\nReference answer: {reference}\nCandidate answer: {answer}\n"
              "Rate how correct the candidate is compared to the reference from 1 to 5. Reply with the number only.")
    reply = U.converse(prompt, model_id=U.LARGE_MODEL, temperature=0, max_tokens=5)
    return float(re.search(r"[1-5]", reply).group())

The tests themselves, also appended.

In [ ]:
%%writefile -a scripts/test_prompt_regression.py


@pytest.mark.parametrize("item", GOLDEN, ids=[g["question"][:40] for g in GOLDEN])
def test_valid_json_and_expected_fact(item):
    parsed, text, _ = ask(item["question"], item["source"])
    assert parsed is not None, f"not valid JSON: {text[:120]}"
    assert {"answer", "grounded"} <= set(parsed), f"missing keys: {parsed}"
    assert parsed["grounded"] is True
    assert numbers(item["answer"]) <= numbers(str(parsed["answer"])), f"fact missing: {parsed['answer']}"


def test_refuses_off_topic():
    parsed, text, _ = ask("Write a poem about football.", "01_coverage_overview.md")
    assert parsed is not None and parsed["grounded"] is False, text[:200]


def test_latency_and_token_budget():
    for item in GOLDEN[:3]:
        _, _, resp = ask(item["question"], item["source"])
        assert resp["metrics"]["latencyMs"] < MAX_LATENCY_MS
        assert resp["usage"]["outputTokens"] <= MAX_OUTPUT_TOKENS


def test_llm_judge_score():
    scores = [judge(i["question"], i["answer"], str((ask(i["question"], i["source"])[0] or {}).get("answer")))
              for i in GOLDEN[:5]]
    assert sum(scores) / len(scores) >= MIN_JUDGE_SCORE, f"judge scores: {scores}"

Run the suite against **version 1**. Look for `passed` / `failed` per test.

In [ ]:
os.environ["PROMPT_ARN"] = prompt_arn_v1
!{sys.executable} -m pytest -q -p no:cacheprovider scripts/test_prompt_regression.py

Run it against **version 2** (the candidate). In a pipeline, a failure here blocks the release.

In [ ]:
os.environ["PROMPT_ARN"] = prompt_arn_v2
!{sys.executable} -m pytest -q -p no:cacheprovider scripts/test_prompt_regression.py

The same checks from the notebook, as a table. Importing the test module reuses its helper functions.

In [ ]:
sys.path.append("scripts")
import test_prompt_regression as T
rows = []
for g in T.GOLDEN[:5]:
    parsed, text, resp = T.ask(g["question"], g["source"])
    rows.append({"question": g["question"], "valid_json": parsed is not None,
                 "fact_ok": parsed is not None and T.numbers(g["answer"]) <= T.numbers(str(parsed.get("answer"))),
                 "latency_ms": resp["metrics"]["latencyMs"], "out_tokens": resp["usage"]["outputTokens"]})
pd.DataFrame(rows)

**Plug it into CI/CD.** A CodeBuild project runs this buildspec as the *Test* stage of a CodePipeline.
A non-zero pytest exit code fails the build and stops the pipeline. The JUnit report shows up in the
CodeBuild *Reports* tab. The CodeBuild role needs `bedrock:InvokeModel` and `bedrock:GetPrompt`.

In [ ]:
%%writefile scripts/buildspec_prompt_tests.yml
version: 0.2
env:
  variables:
    PROMPT_ARN: "arn:aws:bedrock:REGION:ACCOUNT:prompt/PROMPT_ID:2"   # set by the pipeline per release
phases:
  install:
    runtime-versions:
      python: 3.12
    commands:
      - pip install -q -r requirements.txt pytest
  build:
    commands:
      - cd domain3_deployment_orchestration
      - python -m pytest -q scripts/test_prompt_regression.py --junitxml=reports/prompt_tests.xml
reports:
  prompt-regression:
    files:
      - domain3_deployment_orchestration/reports/prompt_tests.xml
    file-format: JUNITXML

## 3.3.8 Agent versions, aliases and deployment pipelines

- **DRAFT**: the working copy. Test it with the built-in alias `TSTALIASID`.
- **Version**: immutable snapshot of a prepared DRAFT. Created when you create an alias **without** a
  routing configuration.
- **Alias**: a named pointer (`live`, `staging`) to a version. Applications call the alias.
- Release = point `live` to the new version. Rollback = point it back. Both are one API call.

Helper to call the agent through an alias.

In [ ]:
def ask_agent(text, alias_id):
    resp = rt.invoke_agent(agentId=agent_id, agentAliasId=alias_id, sessionId=str(uuid.uuid4()), inputText=text)
    return "".join(e["chunk"]["bytes"].decode() for e in resp["completion"] if "chunk" in e)

agent_status = lambda: agent.get_agent(agentId=agent_id)["agent"]["agentStatus"]  # noqa: E731
alias_status = lambda alias_id: agent.get_agent_alias(  # noqa: E731
    agentId=agent_id, agentAliasId=alias_id)["agentAlias"]["agentAliasStatus"]

Where are we now? List the versions and see which one `live` points to.

In [ ]:
live = agent.get_agent_alias(agentId=agent_id, agentAliasId=agent_alias_id)["agentAlias"]
previous_version = live["routingConfiguration"][0]["agentVersion"]
versions = agent.list_agent_versions(agentId=agent_id)["agentVersionSummaries"]
print("versions:", [v["agentVersion"] for v in versions])
print("alias 'live' ->", previous_version)

### Step 1: change the DRAFT
`update_agent` needs the full required configuration again, so read it first. The new instruction
adds a visible marker ("Ref: SafeDrive Claims") so you can tell the versions apart.

In [ ]:
current = agent.get_agent(agentId=agent_id)["agent"]
OLD_INSTRUCTION = current["instruction"]
NEW_INSTRUCTION = OLD_INSTRUCTION.replace(" Always end your answer with 'Ref: SafeDrive Claims'.", "") + \
    " Always end your answer with 'Ref: SafeDrive Claims'."
agent.update_agent(agentId=agent_id, agentName=current["agentName"], foundationModel=current["foundationModel"],
                   agentResourceRoleArn=current["agentResourceRoleArn"], instruction=NEW_INSTRUCTION,
                   idleSessionTTLInSeconds=current["idleSessionTTLInSeconds"])
U.wait_until(agent_status, ok=("NOT_PREPARED", "PREPARED"), bad=("FAILED",), delay=5, label="agent")

### Step 2: prepare the DRAFT

In [ ]:
agent.prepare_agent(agentId=agent_id)
U.wait_until(agent_status, ok=("PREPARED",), bad=("FAILED",), delay=5, label="agent")

### Step 3: create a new version through a `staging` alias
No routing configuration → Bedrock snapshots the DRAFT into a new version and points `staging` to it.
(An old `staging` alias from an earlier run is removed first.)

In [ ]:
for a in agent.list_agent_aliases(agentId=agent_id)["agentAliasSummaries"]:
    if a["agentAliasName"] == "staging":
        U.safe(agent.delete_agent_alias, agentId=agent_id, agentAliasId=a["agentAliasId"])
staging_alias_id = agent.create_agent_alias(
    agentId=agent_id, agentAliasName="staging", description="release candidate")["agentAlias"]["agentAliasId"]
U.wait_until(lambda: alias_status(staging_alias_id), ok=("PREPARED",), bad=("FAILED",), delay=5, label="staging alias")

In [ ]:
staging = agent.get_agent_alias(agentId=agent_id, agentAliasId=staging_alias_id)["agentAlias"]
new_version = staging["routingConfiguration"][0]["agentVersion"]
print("new version:", new_version, "| live still on:", previous_version)

### Step 4: smoke test staging against live
Only the staging answer should end with the marker.

In [ ]:
question = "What is the status of claim C0000001?"
print("live   :", ask_agent(question, agent_alias_id))
print("staging:", ask_agent(question, staging_alias_id))

### Step 5: promote
Point `live` to the new version. Applications keep calling the same alias ID.

In [ ]:
agent.update_agent_alias(agentId=agent_id, agentAliasId=agent_alias_id, agentAliasName="live",
                         routingConfiguration=[{"agentVersion": new_version}])
U.wait_until(lambda: alias_status(agent_alias_id), ok=("PREPARED",), bad=("FAILED",), delay=5, label="live alias")
print("live   :", ask_agent(question, agent_alias_id))

### Step 6: roll back
Something is wrong in production? Repoint `live` to the previous version. The alias keeps a history
of its routing changes for audits.

In [ ]:
agent.update_agent_alias(agentId=agent_id, agentAliasId=agent_alias_id, agentAliasName="live",
                         routingConfiguration=[{"agentVersion": previous_version}])
U.wait_until(lambda: alias_status(agent_alias_id), ok=("PREPARED",), bad=("FAILED",), delay=5, label="live alias")
print("live   :", ask_agent(question, agent_alias_id))

In [ ]:
history = agent.get_agent_alias(agentId=agent_id, agentAliasId=agent_alias_id)["agentAlias"].get("agentAliasHistoryEvents", [])
for h in history:
    print(h.get("startDate"), "->", h.get("endDate"), [r.get("agentVersion") for r in h["routingConfiguration"]])

### AgentCore Runtime versions and endpoints

- `update_agent_runtime` (new code zip or image) creates **version N+1**. Versions are immutable.
- The `DEFAULT` endpoint moves to the latest version automatically. For controlled releases create
  your own endpoint (`prod`) pinned to a version and move it when tests pass.

```python
ac.update_agent_runtime(agentRuntimeId=rid, agentRuntimeArtifact=ARTIFACT, roleArn=ROLE,
                        networkConfiguration={"networkMode": "PUBLIC"})                    # -> version 2
ac.create_agent_runtime_endpoint(agentRuntimeId=rid, name="prod", agentRuntimeVersion="1")
ac.update_agent_runtime_endpoint(agentRuntimeId=rid, endpointName="prod", agentRuntimeVersion="2")  # promote
ac.update_agent_runtime_endpoint(agentRuntimeId=rid, endpointName="prod", agentRuntimeVersion="1")  # roll back
acd.invoke_agent_runtime(agentRuntimeArn=arn, qualifier="prod", ...)       # callers pick the endpoint
```

If you ran 3.1b, list the versions and endpoints of the deployed runtime.

In [ ]:
ac = U.client("bedrock-agentcore-control")
runtime_arn = U.get_state("agentcore_runtime_arn")
if runtime_arn:
    rid = runtime_arn.split("/")[-1]
    vers = ac.list_agent_runtime_versions(agentRuntimeId=rid)["agentRuntimes"]
    eps = ac.list_agent_runtime_endpoints(agentRuntimeId=rid)["runtimeEndpoints"]
    print("versions :", [(v["agentRuntimeVersion"], v["status"]) for v in vers])
    print("endpoints:", [(e["name"], e.get("liveVersion")) for e in eps])
else:
    print("no AgentCore runtime in state (run 3.1b to create one)")

### A deployment pipeline for the agent (sketch)
CodePipeline: **Source** (agent config in Git) → **Build/Test** (prompt tests) → **Deploy to staging**
(script below) → manual approval → **Promote** (`update_agent_alias`). The deploy script is the same
sequence you just ran by hand.

In [ ]:
%%writefile scripts/deploy_agent.py
"""CI step: publish the agent DRAFT as a new version behind a staging alias and smoke test it."""
import os
import sys
import time
import uuid

import boto3

AGENT_ID = os.environ["AGENT_ID"]
agent = boto3.client("bedrock-agent")
rt = boto3.client("bedrock-agent-runtime")


def wait(get_status, ok):
    while (status := get_status()) not in ok:
        if status == "FAILED":
            sys.exit("deployment failed")
        time.sleep(5)


agent.prepare_agent(agentId=AGENT_ID)
wait(lambda: agent.get_agent(agentId=AGENT_ID)["agent"]["agentStatus"], ("PREPARED",))
alias = agent.create_agent_alias(agentId=AGENT_ID, agentAliasName=f"rc-{int(time.time())}")["agentAlias"]
alias_id = alias["agentAliasId"]
wait(lambda: agent.get_agent_alias(agentId=AGENT_ID, agentAliasId=alias_id)["agentAlias"]["agentAliasStatus"],
     ("PREPARED",))
resp = rt.invoke_agent(agentId=AGENT_ID, agentAliasId=alias_id, sessionId=str(uuid.uuid4()),
                       inputText="What is the status of claim C0000001?")
answer = "".join(e["chunk"]["bytes"].decode() for e in resp["completion"] if "chunk" in e)
assert "C0000001" in answer or "status" in answer.lower(), f"smoke test failed: {answer}"
print("release candidate alias:", alias_id)

In [ ]:
%%writefile scripts/buildspec_agent_deploy.yml
version: 0.2
env:
  variables:
    AGENT_ID: "REPLACE_WITH_AGENT_ID"
phases:
  install:
    commands:
      - pip install -q boto3 pytest
  pre_build:
    commands:
      - cd domain3_deployment_orchestration
      - python -m pytest -q scripts/test_prompt_regression.py     # quality gate before any deploy
  build:
    commands:
      - python scripts/deploy_agent.py                             # new version + staging smoke test
  # promotion (update_agent_alias for 'live') runs in a later stage after manual approval

Infrastructure as code: agents, aliases, prompts and knowledge bases also exist as CloudFormation
resources (`AWS::Bedrock::Agent`, `AWS::Bedrock::AgentAlias`, `AWS::Bedrock::Prompt`,
`AWS::Bedrock::PromptVersion`, `AWS::Bedrock::KnowledgeBase`), so a pipeline can deploy them as a stack.

## 3.3.10 FM deployment automation with fine-tuned model versioning 🔒

- A fine-tuning job produces a **custom model**. Bedrock has no built-in version number for it, so use
  a **naming and tagging convention**:

| Item | Convention |
|---|---|
| Job name | `safedrive-claims-qa-ft-<yyyymmdd>` |
| Custom model name | `safedrive-claims-qa-v<N>` |
| Tags | `version=<N>`, `base_model=...`, `dataset=<S3 version or hash>`, `stage=candidate / production` |

- A custom model must be **deployed** before use: **Provisioned Throughput** (hourly, any custom
  model) or a **custom model deployment** for on-demand inference (supported for some models, such as Nova).
- The application reads the model identifier from **SSM Parameter Store**. Promotion = evaluate the
  new version, deploy it, update the parameter. Rollback = set the parameter back.

List the custom models that follow the convention. This is free. The list is empty unless you ran
the 🔒 fine-tuning job in Domain 2.

In [ ]:
models = bedrock.list_custom_models(nameContains="safedrive", sortBy="CreationTime", sortOrder="Descending")["modelSummaries"]
print([(m["modelName"], m["customizationType"], m["creationTime"].date().isoformat()) for m in models])

The config pattern (runs, free): the app reads its model from a parameter. Start with the base model.

In [ ]:
ssm = U.client("ssm")
PARAM = "/safedrive/genai/assistant-model-id"
ssm.put_parameter(Name=PARAM, Value=U.TEXT_MODEL, Type="String", Overwrite=True,
                  Description="Model or deployment ARN used by the SafeDrive assistant")
model_in_use = ssm.get_parameter(Name=PARAM)["Parameter"]["Value"]
print("app uses:", model_in_use)
print(U.converse("Say 'ready' in one word.", model_id=model_in_use))

🔒 **Promote the newest fine-tuned model.** 💰 Needs a finished fine-tuning job. A custom model
deployment bills per token; Provisioned Throughput (commented alternative) bills per hour, so delete it
when you are done. The model version is taken from the name and written as a tag.

In [ ]:
if U.expensive("deploy the latest fine-tuned model") and models:
    latest = models[0]
    deployment_arn = bedrock.create_custom_model_deployment(
        modelDeploymentName=f"{latest['modelName']}-od", modelArn=latest["modelArn"],
        description="on-demand deployment of the newest SafeDrive fine-tuned model",
        tags=[{"key": "project", "value": "mla-c02-practice"}, {"key": "stage", "value": "production"},
              {"key": "model_name", "value": latest["modelName"]}])["customModelDeploymentArn"]
    # Alternative for models without on-demand support (hourly cost):
    # bedrock.create_provisioned_model_throughput(provisionedModelName=..., modelId=latest["modelArn"], modelUnits=1)
    print(deployment_arn)

🔒 Wait for the deployment, then switch the application by updating the parameter.

In [ ]:
if U.expensive("switch the app to the fine-tuned model") and models:
    U.wait_until(lambda: bedrock.get_custom_model_deployment(
        customModelDeploymentIdentifier=deployment_arn)["status"], ok=("Active",), bad=("Failed",), delay=30,
        label="custom model deployment")
    ssm.put_parameter(Name=PARAM, Value=deployment_arn, Type="String", Overwrite=True)
    print(U.converse("What is the standard deductible per claim?", model_id=deployment_arn))

Rollback is a config change. Parameter Store keeps the **history** of values, which is your audit trail.

In [ ]:
ssm.put_parameter(Name=PARAM, Value=U.TEXT_MODEL, Type="String", Overwrite=True)
hist = ssm.get_parameter_history(Name=PARAM)["Parameters"]
pd.DataFrame(hist)[["Version", "Value", "LastModifiedDate"]]

## 3.3.11 RAG updates and Knowledge Base refresh

- The KB does **not** watch S3. A refresh pipeline must call `StartIngestionJob`.
- **Event-driven**: S3 "Object Created" → EventBridge rule → Lambda → `StartIngestionJob`.
- **Scheduled**: EventBridge Scheduler runs the same Lambda every night.
- Only **one ingestion job per data source** can run at a time, so the Lambda must tolerate a
  `ConflictException` when several files arrive together.
- Bigger changes (new chunking, new embedding model): build a second KB, evaluate it, then switch.

### Step 1: the refresh Lambda

In [ ]:
%%writefile scripts/kb_refresh_lambda.py
"""Start a Knowledge Base ingestion job when policy documents change (S3 event or schedule)."""
import os

import boto3

agent = boto3.client("bedrock-agent")
KB_ID, DS_ID = os.environ["KB_ID"], os.environ["DS_ID"]


def handler(event, context):
    key = event.get("detail", {}).get("object", {}).get("key", "scheduled run")
    try:
        job = agent.start_ingestion_job(knowledgeBaseId=KB_ID, dataSourceId=DS_ID,
                                        description=f"refresh after: {key}"[:200])["ingestionJob"]
        print("started", job["ingestionJobId"], "for", key)
        return {"started": True, "ingestionJobId": job["ingestionJobId"]}
    except agent.exceptions.ConflictException:
        print("a sync is already running; it will pick up", key)
        return {"started": False, "reason": "sync already running"}

Create the function `safedrive-kb-refresh` (reuse if it exists).

In [ ]:
import io, zipfile
REFRESH_FN = "safedrive-kb-refresh"
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w") as z:
    z.write("scripts/kb_refresh_lambda.py", "kb_refresh_lambda.py")
try:
    refresh_arn = lam.get_function(FunctionName=REFRESH_FN)["Configuration"]["FunctionArn"]
except lam.exceptions.ResourceNotFoundException:
    refresh_arn = lam.create_function(
        FunctionName=REFRESH_FN, Runtime="python3.12", Role=ROLE, Handler="kb_refresh_lambda.handler",
        Code={"ZipFile": buf.getvalue()}, Timeout=30,
        Environment={"Variables": {"KB_ID": kb_id, "DS_ID": ds_id}},
        Tags={t["Key"]: t["Value"] for t in U.TAGS})["FunctionArn"]
    lam.get_waiter("function_active_v2").wait(FunctionName=REFRESH_FN)
print(refresh_arn)

### Step 2: send S3 events to EventBridge
Turn on EventBridge notifications for the bucket. The existing notification settings are read first
and kept, because this API replaces the whole configuration.

In [ ]:
s3 = U.client("s3")
cfg = s3.get_bucket_notification_configuration(Bucket=U.bucket())
cfg.pop("ResponseMetadata", None)
cfg["EventBridgeConfiguration"] = {}
s3.put_bucket_notification_configuration(Bucket=U.bucket(), NotificationConfiguration=cfg)
print("notification config keys:", list(cfg.keys()))

### Step 3: EventBridge rule
Match new `.md` files under the policy-docs prefix. The metadata sidecar files end in `.json`, so
they do not trigger a second run.

In [ ]:
events = U.client("events")
RULE = "safedrive-kb-refresh-on-upload"
pattern = {"source": ["aws.s3"], "detail-type": ["Object Created"],
           "detail": {"bucket": {"name": [U.bucket()]},
                      "object": {"key": [{"wildcard": f"{C.RAW}/policy_docs/*.md"}]}}}
rule_arn = events.put_rule(Name=RULE, EventPattern=json.dumps(pattern), State="ENABLED",
                           Description="Sync the SafeDrive KB when a policy document is uploaded",
                           Tags=U.TAGS)["RuleArn"]
print(rule_arn)

Add the Lambda as the rule's target and allow EventBridge to invoke it.

In [ ]:
events.put_targets(Rule=RULE, Targets=[{"Id": "kb-refresh", "Arn": refresh_arn}])
U.safe(lam.add_permission, FunctionName=REFRESH_FN, StatementId="eventbridge-invoke",
       Action="lambda:InvokeFunction", Principal="events.amazonaws.com", SourceArn=rule_arn)

### Step 4: test it by uploading a new policy document
Helper: the most recent ingestion job. Remember it before the test upload.

In [ ]:
def latest_job():
    jobs = agent.list_ingestion_jobs(knowledgeBaseId=kb_id, dataSourceId=ds_id, maxResults=1,
                                     sortBy={"attribute": "STARTED_AT", "order": "DESCENDING"})["ingestionJobSummaries"]
    return jobs[0]

job_before = latest_job()["ingestionJobId"]
print("latest job before the upload:", job_before)

Upload the metadata file first, then the document, so one event fires when both are in place.

In [ ]:
NEW_DOC = f"{C.RAW}/policy_docs/06_roadside_assistance.md"
doc = ("# SafeDrive Motor Policy - Roadside Assistance\n\n## Towing\nSafeDrive pays for towing up to 80 km "
       "to the nearest approved repair shop.\n\n## Call-outs\nUp to three roadside call-outs are included "
       "per policy year. Each extra call-out costs USD 45.\n")
meta = {"metadataAttributes": {"category": "roadside", "product": "motor", "year": 2026}}
time.sleep(30)                                   # give the new rule a moment to become active
s3.put_object(Bucket=U.bucket(), Key=NEW_DOC + ".metadata.json", Body=json.dumps(meta))
s3.put_object(Bucket=U.bucket(), Key=NEW_DOC, Body=doc)
print("uploaded", NEW_DOC)

Watch for a **new** ingestion job started by the Lambda (it appears within about a minute).

In [ ]:
for _ in range(18):
    job = latest_job()
    if job["ingestionJobId"] != job_before:
        break
    time.sleep(10)
print("new job:", job["ingestionJobId"] != job_before, "|", job["ingestionJobId"], job["status"], job.get("description"))

In [ ]:
U.wait_until(lambda: latest_job()["status"], ok=("COMPLETE",), bad=("FAILED", "STOPPED"), delay=10, label="ingestion job")
print(latest_job()["statistics"])

Proof: the Knowledge Base can now answer from the new document.

In [ ]:
hits = rt.retrieve(knowledgeBaseId=kb_id, retrievalQuery={"text": "How far will SafeDrive tow my car?"},
                   retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 2}})["retrievalResults"]
for h in hits:
    print(round(h["score"], 3), os.path.basename(h["location"]["s3Location"]["uri"]), "|", h["content"]["text"][:100])

If no job appeared, read the Lambda's log to see whether it was invoked.

In [ ]:
logs = U.client("logs")
try:
    ev = logs.filter_log_events(logGroupName=f"/aws/lambda/{REFRESH_FN}", limit=10)["events"]
    print("".join(e["message"] for e in ev)[-800:])
except Exception as e:  # noqa: BLE001
    print("no logs yet:", str(e)[:120])

### Scheduled refresh with EventBridge Scheduler
A nightly safety net at 02:00 UTC that invokes the same Lambda. (Scheduler can also call
`bedrock-agent:StartIngestionJob` directly as a universal target, without a Lambda.)

In [ ]:
scheduler = U.client("scheduler")
SCHEDULE = "safedrive-kb-nightly-refresh"
U.safe(scheduler.delete_schedule, Name=SCHEDULE)          # make the cell re-runnable
schedule_arn = scheduler.create_schedule(
    Name=SCHEDULE, ScheduleExpression="cron(0 2 * * ? *)", ScheduleExpressionTimezone="UTC",
    FlexibleTimeWindow={"Mode": "OFF"}, State="ENABLED",
    Target={"Arn": refresh_arn, "RoleArn": ROLE, "Input": json.dumps({"source": "nightly-schedule"})})["ScheduleArn"]
print(schedule_arn)

### Step Functions sketch: sync → evaluate → promote 📝
For a controlled refresh, a state machine runs the sync, waits for it, runs an evaluation (for
example the hit-rate check from 3.2b in a Lambda) and only then promotes. The definition below is
printed, not deployed.

In [ ]:
sdk = "arn:aws:states:::aws-sdk:bedrockagent:"
definition = {"StartAt": "StartSync", "States": {
    "StartSync": {"Type": "Task", "Resource": sdk + "startIngestionJob", "ResultPath": "$.sync",
                  "Parameters": {"KnowledgeBaseId": kb_id, "DataSourceId": ds_id}, "Next": "Wait"},
    "Wait": {"Type": "Wait", "Seconds": 30, "Next": "GetSync"},
    "GetSync": {"Type": "Task", "Resource": sdk + "getIngestionJob", "ResultPath": "$.sync",
                "Parameters": {"KnowledgeBaseId": kb_id, "DataSourceId": ds_id,
                               "IngestionJobId.$": "$.sync.IngestionJob.IngestionJobId"}, "Next": "SyncDone?"},
    "SyncDone?": {"Type": "Choice", "Default": "Wait", "Choices": [
        {"Variable": "$.sync.IngestionJob.Status", "StringEquals": "COMPLETE", "Next": "Evaluate"},
        {"Variable": "$.sync.IngestionJob.Status", "StringEquals": "FAILED", "Next": "Failed"}]},
    "Evaluate": {"Type": "Task", "Resource": "arn:aws:states:::lambda:invoke", "ResultPath": "$.eval",
                 "Parameters": {"FunctionName": "safedrive-rag-eval"}, "Next": "GoodEnough?"},
    "GoodEnough?": {"Type": "Choice", "Default": "Failed", "Choices": [
        {"Variable": "$.eval.Payload.hit_rate", "NumericGreaterThanEquals": 0.9, "Next": "Promote"}]},
    "Promote": {"Type": "Task", "Resource": "arn:aws:states:::aws-sdk:ssm:putParameter", "End": True,
                "Parameters": {"Name": "/safedrive/genai/active-kb-id", "Value": kb_id, "Overwrite": True}},
    "Failed": {"Type": "Fail", "Error": "RefreshFailed"}}}
print(json.dumps(definition, indent=1)[:900], "...")

## Exam traps

- An alias pointing at a **version** does not see DRAFT changes. Editing the agent is not a release.
- Rolling back an agent or a prompt means **repointing** (alias, version ARN, parameter), not
  re-editing the draft under pressure.
- Do not assert exact strings in LLM tests. Assert facts, structure and thresholds, and pin
  `temperature=0` in tests.
- A KB sync is **incremental**, but deleted or changed S3 objects only disappear from answers after
  the next ingestion job.
- Hard-coding a custom model ARN in application code makes rollback a code deployment. Keep it in
  Parameter Store or AppConfig.

## Quiz

**1.** A team wants to test a new wording of a system prompt against the current one, keep a history
of changes and let the application pin a tested snapshot. Which feature fits?

A. Guardrails  B. Bedrock Prompt Management with variants and versions  C. Model Registry  D. S3 Versioning of a text file

<details><summary>Answer</summary>B. Variants compare wordings, versions are immutable snapshots the app can pin by ARN.</details>

**2.** After a new Bedrock agent version went live, wrong answers are reported. What is the fastest safe rollback?

A. Edit the DRAFT instruction and prepare again  B. Update the `live` alias routing to the previous version  C. Delete the agent  D. Lower the temperature

<details><summary>Answer</summary>B. Versions are immutable; repointing the alias restores the old behavior at once.</details>

**3.** Policy documents are uploaded to S3 at random times and must be searchable within minutes.
Which design needs the least custom work?

A. A cron job on EC2 that rebuilds the KB  B. S3 event → EventBridge → Lambda calling `StartIngestionJob`  C. Recreate the KB after each upload  D. Increase `numberOfResults`

<details><summary>Answer</summary>B. Event-driven sync; the ingestion job is incremental.</details>

**4.** Which check belongs in a prompt regression suite that runs in CodeBuild on every prompt change?

A. Exact string equality with last week's answers  B. Golden questions with fact, format and judge-score thresholds  C. A manual review of 500 answers  D. GPU utilization

<details><summary>Answer</summary>B. Automated, tolerant assertions; exact matches break on harmless wording changes.</details>

**5.** A new fine-tuned model version must replace the current one with an easy rollback and no code
change. What should the pipeline do after evaluation passes?

A. Rename the old custom model  B. Deploy the new model and update an SSM parameter that the app reads  C. Rebuild the application container  D. Change the model in every prompt by hand

<details><summary>Answer</summary>B. The model identifier is configuration; rollback is setting the old value again.</details>

## Save state

In [ ]:
state = U.save_state(prompt_id=prompt_id)
print({k: state.get(k) for k in ("prompt_id", "agent_id", "agent_alias_id", "kb_id")})

## Cleanup

**Kept for later notebooks:** the managed prompt `safedrive-policy-answer` (no cost), the agent with
alias `live` (back on its original version), the Knowledge Base, and EventBridge notifications on the
bucket (harmless). The teardown notebook removes them.

Deleted here: the refresh trigger, schedule and Lambda, the test document, the staging alias and
version, and the SSM parameter.

### Before you delete: what later notebooks use

The cleanup cells below **do not delete** the items in this table.

| Kept | Used by later notebooks | If you delete it anyway |
|---|---|---|
| Agent with alias `live`, Lambda, Knowledge Base | 4.1b, 4.2, 4.3b | Rebuilt by the catch-up cell |
| Managed prompt `safedrive-policy-answer` | Not used again | - |

- **Continuing to the next notebook soon?** Run the cleanup cells and keep the items above. None of them bills while idle (only a little storage).
- **Stopping for good?** Run `99_cleanup/99_teardown.ipynb`; it removes everything, including these.

In [ ]:
U.safe(scheduler.delete_schedule, Name=SCHEDULE)

In [ ]:
U.safe(events.remove_targets, Rule=RULE, Ids=["kb-refresh"])
U.safe(events.delete_rule, Name=RULE)

In [ ]:
U.safe(lam.delete_function, FunctionName=REFRESH_FN)

Remove the test document and sync once more, so the Knowledge Base holds the original five documents.

In [ ]:
U.safe(s3.delete_object, Bucket=U.bucket(), Key=NEW_DOC)
U.safe(s3.delete_object, Bucket=U.bucket(), Key=NEW_DOC + ".metadata.json")
U.safe(C.sync_kb, kb_id, ds_id)

Agent: remove the staging alias and its version, then restore the original DRAFT instruction.

In [ ]:
U.safe(agent.delete_agent_alias, agentId=agent_id, agentAliasId=staging_alias_id)
time.sleep(10)
U.safe(agent.delete_agent_version, agentId=agent_id, agentVersion=new_version, skipResourceInUseCheck=True)

In [ ]:
U.safe(agent.update_agent, agentId=agent_id, agentName=current["agentName"],
       foundationModel=current["foundationModel"], agentResourceRoleArn=current["agentResourceRoleArn"],
       instruction=OLD_INSTRUCTION, idleSessionTTLInSeconds=current["idleSessionTTLInSeconds"])
time.sleep(10)
U.safe(agent.prepare_agent, agentId=agent_id)

In [ ]:
U.safe(ssm.delete_parameter, Name=PARAM)

🔒 Only if you ran the fine-tuned model deployment: delete it to stop any cost.

In [ ]:
if "deployment_arn" in globals():
    U.safe(bedrock.delete_custom_model_deployment, customModelDeploymentIdentifier=deployment_arn)